# Sinh bài — Baseline (Azure OpenAI)

Sinh `output` cho từng case qua Azure OpenAI, **không chạy eval**.

**Input:** `dataset/fnb_dataset_test.json`  
**Output:** `results/generations/baseline_MODEL_TIMESTAMP.json`

Sau khi sinh xong, đưa file JSON vào `evaluation/evaluate.ipynb` để chạy metrics.

In [1]:
# %pip install -q openai pandas python-dotenv

In [2]:
import os
import re
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
load_dotenv(_dotenv_path, override=True, encoding="utf-8")
ROOT = Path(_dotenv_path).resolve().parent if _dotenv_path else Path.cwd().resolve()

# ===== Hyperparameters =====
DATASET_REL          = "dataset/fnb_dataset_test.json"
OUTPUT_REL           = "results/generations"
GENERATION_TEMPERATURE = 0.4

# Azure OpenAI settings
def _env(k: str) -> str:
    return (os.getenv(k) or "").strip()

BASELINE_MODEL             = (_env("BASELINE_MODEL") or "gpt-4o-mini").strip()
BASELINE_AZURE_ENDPOINT    = (_env("BASELINE_AZURE_ENDPOINT") or _env("AZURE_OPENAI_ENDPOINT")).rstrip("/")
BASELINE_AZURE_API_KEY     = _env("BASELINE_AZURE_API_KEY") or _env("AZURE_OPENAI_API_KEY")
BASELINE_AZURE_API_VERSION = _env("BASELINE_AZURE_API_VERSION") or _env("OPENAI_API_VERSION") or "2024-08-01-preview"

if not (BASELINE_AZURE_ENDPOINT and BASELINE_AZURE_API_KEY):
    raise RuntimeError("Cần BASELINE_AZURE_ENDPOINT + BASELINE_AZURE_API_KEY (hoặc AZURE_OPENAI_*).")

MAX_RETRIES = 3

DATASET_PATH = ROOT / DATASET_REL
OUTPUT_DIR   = ROOT / OUTPUT_REL

VN_TZ      = timezone(timedelta(hours=7))
RUN_ID     = datetime.now(VN_TZ).strftime("%d-%m-%Y_%H-%M")
safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", BASELINE_MODEL).strip("_") or "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / f"{safe_model}_{RUN_ID}.json"

print(f"Model  : {BASELINE_MODEL} @ {BASELINE_AZURE_ENDPOINT}")
print(f"Input  : {DATASET_PATH}")
print(f"Output : {output_path}")

Model  : md-gpt-5.4-mini @ https://vqnhan-poc.openai.azure.com
Input  : D:\Github\mcs-train-content-model\dataset\fnb_dataset_test.json
Output : D:\Github\mcs-train-content-model\results\generations\md-gpt-5.4-mini_21-05-2026_14-31.json


In [3]:
from openai import AzureOpenAI

_client = AzureOpenAI(
    azure_endpoint=BASELINE_AZURE_ENDPOINT,
    api_key=BASELINE_AZURE_API_KEY,
    api_version=BASELINE_AZURE_API_VERSION,
)

_r = _client.chat.completions.create(
    model=BASELINE_MODEL,
    messages=[{"role": "user", "content": "Xin chào"}],
    temperature=GENERATION_TEMPERATURE,
)
print("Baseline OK:", (_r.choices[0].message.content or "").strip()[:100])

Baseline OK: Xin chào! Mình có thể giúp gì cho bạn hôm nay?


In [4]:
import json
import time

BOLD  = "\033[1m"
CYAN  = "\033[36m"
RESET = "\033[0m"


def generate_azure_chat(messages, model=None, verbose=True):
    """
    Gọi Azure OpenAI chat completions (non-streaming).
    Trả về dict:
        answer          : str   — nội dung trả lời
        time_generation : float — giây sinh output (wall clock)
        tokens_input    : int   — prompt tokens
        tokens_output   : int   — completion tokens
        tokens_total    : int   — tokens_input + tokens_output
    """
    model = model or BASELINE_MODEL

    t0 = time.perf_counter()
    r  = _client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=GENERATION_TEMPERATURE,
    )
    elapsed = time.perf_counter() - t0

    content = (r.choices[0].message.content or "").strip()

    usage        = r.usage
    tokens_input  = usage.prompt_tokens     if usage else 0
    tokens_output = usage.completion_tokens if usage else 0
    tokens_total  = tokens_input + tokens_output
    tps = tokens_output / elapsed if elapsed > 0 else 0

    if verbose:
        print(f"{BOLD}[Trả lời]{RESET}  (gen: {elapsed:.1f}s)")
        print(content)
        print(
            f"\n{CYAN}── total: {elapsed:.1f}s | in: {tokens_input} tok"
            f" | out: {tokens_output} tok | {tps:.1f} tok/s ──{RESET}"
        )

    return {
        "answer":          content,
        "time_generation": round(elapsed, 3),
        "tokens_input":    tokens_input,
        "tokens_output":   tokens_output,
        "tokens_total":    tokens_total,
    }


print("generate_azure_chat: sẵn sàng.")

generate_azure_chat: sẵn sàng.


In [5]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

if not isinstance(dataset, list):
    raise ValueError("Dataset phải là mảng JSON.")

missing_ids = [i for i, r in enumerate(dataset) if "id" not in r]
if missing_ids:
    raise ValueError(f"Thiếu field 'id' ở {len(missing_ids)} record: indices {missing_ids[:5]}")

print(f"Loaded {len(dataset)} records từ {DATASET_PATH.name}")
print(f"ID range: {dataset[0]['id']} → {dataset[-1]['id']}")

Loaded 100 records từ fnb_dataset_test.json
ID range: 1 → 100


In [6]:
# Demo 1 bài mẫu để xác nhận chất lượng trước khi chạy batch
_demo = dataset[13]
_user_msg = (
    f"Viết bài marketing Markdown từ tiêu đề và mồi:\n\n"
    f"Tiêu đề: {_demo['title']}\n\n"
    f"Mồi:\n{_demo['seed']}"
)

print("=" * 60)
print("[SYSTEM PROMPT]")
print(_demo["instruction"])
print()
print("[USER PROMPT]")
print(_user_msg)
print("=" * 60)

generate_azure_chat(
    messages=[
        {"role": "system", "content": _demo["instruction"]},
        {"role": "user",   "content": _user_msg},
    ],
)

[SYSTEM PROMPT]
Bạn là copywriter marketing người Việt, giọng director marketing F&B dày dạn, viết Facebook Ads native, rõ offer, ngắn gọn và có CTA chốt đơn. Bám sát seed, ưu tiên hook bán hàng, không hype rỗng, không bịa claim ngoài dữ liệu đã cho.

[USER PROMPT]
Viết bài marketing Markdown từ tiêu đề và mồi:

Tiêu đề: Viết caption Facebook Ads cho Seoul Fire BBQ — buffet lẩu nướng 199K, hook đi nhóm càng lời.

Mồi:
Thương hiệu: Seoul Fire BBQ — chuỗi lẩu nướng, 8 điểm bán Hà Nội
Sản phẩm: Buffet Seoul 199K — 40+ món thịt bò, heo, gà, panchan
Giá: 199.000đ (gốc 269.000đ, -26%), tặng Pepsi refill
Điều kiện: 17h-22h T2-T5, đặt inbox FB, áp dụng 4 người trở lên
Đối tượng: 22-35 tuổi, dân văn phòng, Cầu Giấy, Đống Đa, Thanh Xuân
Kênh: Facebook Page, Messenger, Zalo OA
KPI: CTR ≥ 2,8%, CPL ≤ 18.000đ, ROAS ≥ 4
[Trả lời]  (gen: 2.1s)
## Đi nhóm càng lời — Buffet Seoul 199K tại Seoul Fire BBQ

**40+ món thịt bò, heo, gà, panchan**  
Chỉ **199.000đ/người** thay vì **269.000đ**  
**Giảm 26%** 

{'answer': '## Đi nhóm càng lời — Buffet Seoul 199K tại Seoul Fire BBQ\n\n**40+ món thịt bò, heo, gà, panchan**  \nChỉ **199.000đ/người** thay vì **269.000đ**  \n**Giảm 26%** + **tặng Pepsi refill**\n\nĐi **4 người trở lên** là đã thấy “lời” rõ:  \n- Nhiều món để chọn, ăn no đúng kiểu buffet  \n- Giá tốt cho nhóm bạn, đồng nghiệp, team văn phòng  \n- Áp dụng khung giờ **17h–22h, Thứ 2–Thứ 5**\n\n**Seoul Fire BBQ** có **8 điểm bán tại Hà Nội** — tiện ghé cho dân **Cầu Giấy, Đống Đa, Thanh Xuân**.\n\n👉 **Inbox fanpage ngay** để giữ bàn  \nHoặc nhắn **Messenger / Zalo OA**  \n**Buffet Seoul 199K** — đi nhóm càng đáng tiền.\n\n#SeoulFireBBQ #Buffet199K #LauNuongHaNoi #AnLaLoi #DiNhomCangLoi',
 'time_generation': 2.085,
 'tokens_input': 285,
 'tokens_output': 248,
 'tokens_total': 533}

In [7]:
def build_user_prompt(title: str, seed: str) -> str:
    return (
        f"Viết bài marketing Markdown từ tiêu đề và mồi:\n\n"
        f"Tiêu đề: {title}\n\n"
        f"Mồi:\n{seed}"
    )

def _save(path, recs):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(recs, f, ensure_ascii=False, indent=2)

# --- Resume: nạp bản ghi đã có nếu file output tồn tại ---
records  = []
seen_ids = set()
if output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        records = json.load(f)
    seen_ids = {r["id"] for r in records if r.get("id")}
    print(f"Resume: tìm thấy {len(records)} bản ghi đã có trong {output_path.name}")

_t0 = time.perf_counter()

for idx, row in enumerate(dataset, start=1):
    row_id = row["id"]

    if row_id in seen_ids:
        print(f"[{idx}/{len(dataset)}] Bỏ qua (đã có id={row_id}): {row['title'][:55]!r}")
        continue

    elapsed_total = time.perf_counter() - _t0
    done  = len(records)
    empty = sum(1 for r in records if not r["output"])
    print(
        f"\n[{idx}/{len(dataset)}] Đang xử lý: {row['title'][:60]!r}"
        f" | xong: {done} ({empty} rỗng) | tổng: {elapsed_total:.1f}s",
        flush=True,
    )

    for attempt in range(MAX_RETRIES):
        try:
            r = generate_azure_chat(
                messages=[
                    {"role": "system", "content": row["instruction"]},
                    {"role": "user",   "content": build_user_prompt(row["title"], row["seed"])},
                ],
                verbose=False,
            )
            rec = {
                "id":              row_id,
                "instruction":     row["instruction"],
                "title":           row["title"],
                "seed":            row["seed"],
                "output":          r["answer"],
                "time_generation": r["time_generation"],
                "tokens_input":    r["tokens_input"],
                "tokens_output":   r["tokens_output"],
                "tokens_total":    r["tokens_total"],
            }
            records.append(rec)
            seen_ids.add(row_id)
            _save(output_path, records)          # lưu ngay sau mỗi bài
            print(
                f"  ✓ id={row_id}"
                f" | gen: {r['time_generation']:.1f}s"
                f" | out: {r['tokens_output']} tok",
                flush=True,
            )
            break
        except Exception as e:
            print(f"  Lỗi lần {attempt + 1}/{MAX_RETRIES}: {e!r}", flush=True)
            if attempt + 1 == MAX_RETRIES:
                rec = {
                    "id":              row_id,
                    "instruction":     row["instruction"],
                    "title":           row["title"],
                    "seed":            row["seed"],
                    "output":          "",
                    "time_generation": 0.0,
                    "tokens_input":    0,
                    "tokens_output":   0,
                    "tokens_total":    0,
                }
                records.append(rec)
                seen_ids.add(row_id)
                _save(output_path, records)      # lưu ngay kể cả khi lỗi


[1/100] Đang xử lý: 'Viết caption Facebook cho landing page ưu đãi của Bếp Quê OC' | xong: 0 (0 rỗng) | tổng: 0.0s
  ✓ id=1 | gen: 2.2s | out: 245 tok

[2/100] Đang xử lý: 'Viết caption Facebook cho chiến dịch Trung Thu của Matcha Mâ' | xong: 1 (0 rỗng) | tổng: 2.2s
  ✓ id=2 | gen: 3.8s | out: 490 tok

[3/100] Đang xử lý: 'Viết caption Facebook cho activation tại điểm bán của Cầu Tr' | xong: 2 (0 rỗng) | tổng: 6.0s
  ✓ id=3 | gen: 2.2s | out: 283 tok

[4/100] Đang xử lý: 'Viết post Facebook báo cáo hiệu quả chiến dịch combo trưa củ' | xong: 3 (0 rỗng) | tổng: 8.2s
  ✓ id=4 | gen: 2.4s | out: 330 tok

[5/100] Đang xử lý: "Viết caption Facebook cho Herb n' Glow Detox Tea — combo 7 n" | xong: 4 (0 rỗng) | tổng: 10.6s
  ✓ id=5 | gen: 3.9s | out: 497 tok

[6/100] Đang xử lý: 'Viết caption Instagram cho Urban Brew Coffee — Cold Brew Oat' | xong: 5 (0 rỗng) | tổng: 14.6s
  ✓ id=6 | gen: 2.0s | out: 204 tok

[7/100] Đang xử lý: 'Viết caption Facebook A/B test cho Cơm Tấm Trưa Nay — suất c' | 

In [8]:
done  = [r for r in records if r["output"]]
empty = [r for r in records if not r["output"]]

def _avg(recs, key):
    vals = [r[key] for r in recs if key in r]
    return sum(vals) / len(vals) if vals else 0.0

print(f"Hoàn tất: {len(records)} bản ghi ({len(empty)} rỗng) đã lưu vào: {output_path}")
if done:
    print()
    print(f"{'Thống kê trung bình':─<45} (n={len(done)} bài thành công)")
    print(f"  time_generation : {_avg(done, 'time_generation'):>8.2f} s")
    print(f"  tokens_input    : {_avg(done, 'tokens_input'):>8.1f} tok")
    print(f"  tokens_output   : {_avg(done, 'tokens_output'):>8.1f} tok")
    print(f"  tokens_total    : {_avg(done, 'tokens_total'):>8.1f} tok")
print()
print("Bước tiếp theo: mở evaluation/evaluate.ipynb và đặt:")
print(f'  INPUT_JSON = ROOT / "{output_path.relative_to(ROOT)}"')
print(f'  MODEL_ROLE = "baseline"')
print(f'  MODEL_ID   = "{BASELINE_MODEL}"')

Hoàn tất: 100 bản ghi (0 rỗng) đã lưu vào: D:\Github\mcs-train-content-model\results\generations\md-gpt-5.4-mini_21-05-2026_14-31.json

Thống kê trung bình────────────────────────── (n=100 bài thành công)
  time_generation :     3.18 s
  tokens_input    :    277.5 tok
  tokens_output   :    398.7 tok
  tokens_total    :    676.1 tok

Bước tiếp theo: mở evaluation/evaluate.ipynb và đặt:
  INPUT_JSON = ROOT / "results\generations\md-gpt-5.4-mini_21-05-2026_14-31.json"
  MODEL_ROLE = "baseline"
  MODEL_ID   = "md-gpt-5.4-mini"
